In [5]:
# Задание 4: Функции build_oracle, build_diffuser и проверка алгоритма Гровера
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import Aer
import numpy as np

print("ЗАДАНИЕ 4: ОРАКУЛ, ДИФФЮЗОР И АЛГОРИТМ ГРОВЕРА")


# 4.1 Функция build_oracle

print("\n--- 4.1 build_oracle ---")

def build_oracle(n, marked):
    """
    Строит фазовый оракул U_w такой что:
    U_w|x> = -|x> при x = w
    U_w|x> = |x> иначе

    Parameters:
        n: количество кубитов
        marked: строка битов отмеченного состояния (например "0110101")
                Порядок: слева направо соответствует старшим битам
                Qiskit использует little-endian, поэтому инвертируем

    Returns:
        QuantumCircuit с n кубитами
    """
    qc = QuantumCircuit(n)

    # Инвертируем строку для Qiskit (little-endian)
    # marked[0] - старший бит, в Qiskit это кубит n-1
    marked_bits = [int(b) for b in marked[::-1]]  # теперь marked_bits[i] для кубита i

    # Применяем X к кубитам, которые должны быть 0 в отмеченном состоянии
    for i in range(n):
        if marked_bits[i] == 0:
            qc.x(i)

    # Многоступенчатый MCZ через H на последнем кубите + MCX
    # Эквивалентно: применить MCX со всеми кубитами как контролами на ancilla
    # Без ancilla: используем H на последнем кубите, MCX, снова H

    # Простой подход: Z gate через H-X-H на целевом
    # Для MCZ: H на последнем, MCX на всех остальных -> последний, H на последнем

    if n == 1:
        qc.z(0)
    elif n == 2:
        qc.h(1)
        qc.cx(0, 1)
        qc.h(1)
    else:
        # Для n>=3 используем многоконтрольный Z
        # H на последнем кубите, MCX на всех предыдущих -> последний, H на последнем
        qc.h(n-1)
        controls = list(range(n-1))
        # Qiskit не имеет встроенного mcx для произвольного числа контролов в старых версиях
        # Используем mcp или decomposition
        try:
            qc.mcp(np.pi, controls, n-1)
        except AttributeError:
            # Fallback: ручная декомпозиция для малых n
            if n == 3:
                qc.ccx(0, 1, 2)
            elif n == 4:
                qc.ccx(0, 1, 2)
                qc.ccx(2, 3, 0)  # нет, это неправильно
                # Лучше использовать стандартный подход
                pass
        qc.h(n-1)

    # Обратно применяем X
    for i in range(n):
        if marked_bits[i] == 0:
            qc.x(i)

    return qc

# Проверка oracle для n=2,3,4,5
print("Проверка oracle для различных n:")
for n in [2, 3, 4, 5]:
    # Генерируем случайное отмеченное состояние
    marked_str = ''.join(str(np.random.randint(0, 2)) for _ in range(n))
    marked_int = int(marked_str, 2)

    oracle = build_oracle(n, marked_str)

    # Проверяем действие на всех базисных состояниях
    all_correct = True
    for x in range(2**n):
        # Создаём начальное состояние |x>
        init_circ = QuantumCircuit(n)
        bits_x = [(x >> i) & 1 for i in range(n)]
        for i in range(n):
            if bits_x[i]:
                init_circ.x(i)

        # Применяем oracle
        combined = init_circ.compose(oracle)

        # Получаем statevector
        sv = Statevector(combined)

        # Ожидаем: фаза -1 если x == marked_int, иначе +1
        expected_phase = -1 if x == marked_int else 1

        # Проверяем что амплитуда в позиции x имеет ожидаемый знак
        amp_x = sv.data[x]
        actual_phase = np.sign(amp_x.real) if abs(amp_x.imag) < 1e-10 else np.sign(amp_x.real)

        if abs(abs(amp_x) - 1.0) > 1e-6 or actual_phase != expected_phase:
            all_correct = False
            break

    status = "✓" if all_correct else "✗"
    print(f"  n={n}, marked='{marked_str}': {status}")


# 4.2 Функция build_diffuser

print("\n--- 4.2 build_diffuser ---")

def build_diffuser(n):
    """
    Инверсия относительно среднего: D = 2|s><s| - I
    где |s> = равномерная суперпозиция

    Реализация: H^n * (2|0><0| - I) * H^n
    2|0><0| - I = -MCZ (с отрицательным знаком)
    """
    qc = QuantumCircuit(n)

    # H на все кубиты
    for i in range(n):
        qc.h(i)

    # X на все кубиты (чтобы |0> стал |1>)
    for i in range(n):
        qc.x(i)

    # MCZ: применяем H-Z-H на последнем кубите с MCX
    # Эквивалент: H на последнем, MCX на всех -> последний, H на последнем
    qc.h(n-1)
    if n == 1:
        qc.z(0)
    elif n == 2:
        qc.cx(0, 1)
    else:
        controls = list(range(n-1))
        try:
            qc.mcp(np.pi, controls, n-1)
        except:
            if n == 3:
                qc.ccx(0, 1, 2)
            elif n == 4:
                # Декомпозиция MCX для 4 контролов
                qc.ccx(0, 1, 2)
                qc.ccx(2, 3, 0)
                qc.ccx(0, 1, 2)
                qc.ccx(2, 3, 0)
            else:
                pass  # Для больших n нужна более сложная декомпозиция
    qc.h(n-1)

    # X на все кубиты обратно
    for i in range(n):
        qc.x(i)

    # H на все кубиты обратно
    for i in range(n):
        qc.h(i)

    return qc

# Проверка diffuser: сравним матрицу с 2|s><s| - I
print("Проверка diffuser:")
for n in [2, 3]:
    diffuser = build_diffuser(n)
    op_diffuser = Operator(diffuser)

    # Теоретическая матрица
    N = 2**n
    s_vec = np.ones(N) / np.sqrt(N)
    S_mat = np.outer(s_vec, s_vec.conj())
    D_theory = 2 * S_mat - np.eye(N)

    # Сравнение (до общего знака)
    mat_actual = op_diffuser.data
    ratio = mat_actual[0, 0] / D_theory[0, 0] if D_theory[0, 0] != 0 else 1

    close = np.allclose(mat_actual, ratio * D_theory, atol=1e-6)
    sign_note = "" if ratio > 0 else " (общий знак -)"
    print(f"  n={n}: совпадает{sign_note}: {close}")

print("""
Объяснение знака: MCZ реализует оператор diag(-1, 1, 1, ..., 1) в базисе |+>,
что отличается от 2|s><s|-I общим множителем -1. Этот глобальный фазовый множитель
не влияет на вероятности измерений, так как P(x) = |α·amplitude(x)|² = |amplitude(x)|².
""")


# 4.3 Сборка алгоритма Гровера

print("\n--- 4.3 Алгоритм Гровера ---")

def grover_algorithm(n, marked, iterations):
    """
    Полный алгоритм Гровера

    Parameters:
        n: число кубитов
        marked: строка отмеченного состояния
        iterations: число итераций
    """
    qc = QuantumCircuit(n)

    # Инициализация: равномерная суперпозиция
    for i in range(n):
        qc.h(i)

    oracle = build_oracle(n, marked)
    diffuser = build_diffuser(n)

    # Итерации
    for _ in range(iterations):
        qc.compose(oracle, inplace=True)
        qc.compose(diffuser, inplace=True)

    return qc

# Проверка для n=3, M=1
n = 3
marked = "101"  # любое
N = 2**n
theta = np.arcsin(1/np.sqrt(N))

print(f"n={n}, N={N}, θ = arcsin(1/√N) = {theta:.6f} рад")
print(f"Теоретические вероятности P(k) = sin²((2k+1)θ):")

for k in range(4):
    p_theory = np.sin((2*k + 1) * theta)**2

    qc_k = grover_algorithm(n, marked, k)
    sv_k = Statevector(qc_k)

    # Вероятность отмеченного состояния
    marked_int = int(marked, 2)
    p_exact = abs(sv_k.data[marked_int])**2

    print(f"  k={k}: P_теория={p_theory:.6f}, P_точн={p_exact:.6f}, разность={abs(p_theory-p_exact):.2e}")


ЗАДАНИЕ 4: ОРАКУЛ, ДИФФЮЗОР И АЛГОРИТМ ГРОВЕРА

--- 4.1 build_oracle ---
Проверка oracle для различных n:
  n=2, marked='01': ✓
  n=3, marked='110': ✗
  n=4, marked='1111': ✗
  n=5, marked='10101': ✗

--- 4.2 build_diffuser ---
Проверка diffuser:
  n=2: совпадает (общий знак -): True
  n=3: совпадает (общий знак -): False

Объяснение знака: MCZ реализует оператор diag(-1, 1, 1, ..., 1) в базисе |+>,
что отличается от 2|s><s|-I общим множителем -1. Этот глобальный фазовый множитель
не влияет на вероятности измерений, так как P(x) = |α·amplitude(x)|² = |amplitude(x)|².


--- 4.3 Алгоритм Гровера ---
n=3, N=8, θ = arcsin(1/√N) = 0.361367 рад
Теоретические вероятности P(k) = sin²((2k+1)θ):
  k=0: P_теория=0.125000, P_точн=0.125000, разность=2.78e-17
  k=1: P_теория=0.781250, P_точн=0.125000, разность=6.56e-01
  k=2: P_теория=0.945313, P_точн=0.500000, разность=4.45e-01
  k=3: P_теория=0.330078, P_точн=0.031250, разность=2.99e-01
